# Web Search Tool

Claude includes a built-in web search tool that lets it search the internet for current or specialized information to answer user questions. Unlike other tools where you need to provide the implementation, Claude handles the entire search process automatically - you just need to provide a simple schema to enable it.

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

In [4]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle?
    """,
)
response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_013vCH41WRYceudNUu7AUchs', container=None, content=[TextBlock(citations=None, text='The best exercises for gaining leg muscle are compound movements that target multiple muscle groups. Here are the top choices:\n\n## **Squats** (Back Squats or Front Squats)\nWidely considered the king of leg exercises, squats work your quadriceps, hamstrings, glutes, and calves all at once. They allow you to lift heavy weight and progressively overload, which is crucial for muscle growth.\n\n## **Deadlifts** (Conventional or Romanian)\nDeadlifts primarily target the posterior chain - hamstrings, glutes, and lower back. Romanian deadlifts specifically emphasize hamstring development.\n\n## **Lunges** (Walking, Reverse, or Bulgarian Split Squats)\nThese unilateral exercises help correct muscle imbalances and work each leg independently while engaging stabilizer muscles. Bulgarian split squats are particularly effective for quad and glute development.\n\n## **Leg Press**\nWhile not as func

# How the Response Works
When Claude uses the web search tool, the response contains several types of blocks:

- Text blocks - Claude's explanation of what it's doing
- ServerToolUseBlock - Shows the exact search query Claude used
- WebSearchToolResultBlock - Contains the search results
- WebSearchResultBlock - Individual search results with titles and URLs
- Citation blocks - Text that supports Claude's statements

You can limit searches to specific domains using the **allowed_domains** field. This is particularly useful when you want reliable, authoritative sources

# Practical Usage
The web search tool works best for:

- Current events and recent developments
- Specialized information not in Claude's training data
- Fact-checking and finding authoritative sources
- Research tasks requiring up-to-date information
- Simply include the schema in your tools array when making API calls, and Claude will automatically decide when a web search would help answer the user's question.

